# 02 - Candidate Variable Acquisition
Main goal: Use our raw data to calculate our candidate variables

In [35]:
import pandas as pd
import numpy as np
import os
from scipy.stats import mstats
os.chdir('/Users/thomas/Documents/GitHub/sv_project')

In [36]:
spx = pd.read_csv('data/raw/01_spx_returns.csv')
spx = spx[['Date', 'Returns']]
spx

,Date,Returns
0,1996-01-03,0.000951
1,1996-01-04,-0.005826
2,1996-01-05,-0.001603
3,1996-01-08,0.002838
4,1996-01-09,-0.014568
...,...,...
7650,2026-05-29,0.002172
7651,2026-06-01,0.002625
7652,2026-06-02,0.001292
7653,2026-06-03,-0.007372


In [40]:
#Return-based volatility and downside-risk variables

#Annualized square return
spx['r2_ann'] = (spx['Returns']**2)*252
spx['r2_ann_winsorized'] = mstats.winsorize(spx['r2_ann'], limits=[0.0, 0.005])

#Rolling RV
for window in [5, 21, 63, 126, 252]:
    spx[f'RV_{window}'] = spx['r2_ann_winsorized'].rolling(window).mean()

#EWMA
for lam in [0.94, 0.97, 0.985]:
    spx[f'EWMA_lam_{lam}'] = spx['r2_ann_winsorized'].ewm(alpha = 1-lam, adjust = False).mean()

#Downside RV
spx['RV_down'] = spx['r2_ann_winsorized'].where(spx['Returns'] < 0, 0)
for window in [5, 21, 63]: 
    spx[f'RV_down_{window}'] = spx['RV_down'].rolling(window).mean()

#Return-shape controls
spx['abs_ret_norm'] = spx['Returns'].abs() * np.sqrt(252)
spx['neg_ret_dummy'] = np.where(spx['Returns'] < 0, 1, 0)
spx['neg_ret_mag'] = spx['Returns'].where(spx['Returns'] < 0, 0).abs()
spx

,Date,Returns,r2_ann,r2_ann_winsorized,RV_5,RV_21,RV_63,RV_126,RV_252,EWMA_lam_0.94,EWMA_lam_0.97,EWMA_lam_0.985,RV_down,RV_down_5,RV_down_21,RV_down_63,abs_ret_norm,neg_ret_dummy,neg_ret_mag
0,1996-01-03,0.000951,0.000228,0.000228,NaN,NaN,NaN,NaN,NaN,0.000228,0.000228,0.000228,0.000000,NaN,NaN,NaN,0.015089,0,0.000000
1,1996-01-04,-0.005826,0.008554,0.008554,NaN,NaN,NaN,NaN,NaN,0.000727,0.000477,0.000353,0.008554,NaN,NaN,NaN,0.092490,1,0.005826
2,1996-01-05,-0.001603,0.000647,0.000647,NaN,NaN,NaN,NaN,NaN,0.000722,0.000483,0.000357,0.000647,NaN,NaN,NaN,0.025442,1,0.001603
3,1996-01-08,0.002838,0.002029,0.002029,NaN,NaN,NaN,NaN,NaN,0.000801,0.000529,0.000382,0.000000,NaN,NaN,NaN,0.045046,0,0.000000
4,1996-01-09,-0.014568,0.053484,0.053484,0.012989,NaN,NaN,NaN,NaN,0.003962,0.002118,0.001179,0.053484,0.012537,NaN,NaN,0.231267,1,0.014568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7650,2026-05-29,0.002172,0.001189,0.001189,0.004488,0.011994,0.021251,0.016356,0.014299,0.013427,0.016672,0.017746,0.000000,0.000000,0.002778,0.007701,0.034484,0,0.000000
7651,2026-06-01,0.002625,0.001737,0.001737,0.004136,0.010819,0.021278,0.016274,0.014290,0.012726,0.016224,0.017506,0.000000,0.000000,0.002778,0.007701,0.041675,0,0.000000
7652,2026-06-02,0.001292,0.000421,0.000421,0.002339,0.010736,0.020928,0.016220,0.014292,0.011987,0.015750,0.017250,0.000000,0.000000,0.002778,0.007344,0.020511,0,0.000000
7653,2026-06-03,-0.007372,0.013695,0.013695,0.005077,0.011190,0.020904,0.016272,0.014329,0.012090,0.015688,0.017197,0.013695,0.002739,0.003232,0.007561,0.117028,1,0.007372
